# Fact Knowledge Layer — Colab (T4) Run

This notebook mirrors the full web app (Streamlit + FastAPI): **upload → extract → stats → compare → four demo cases**, plus packaging for the assignment.

Run **all** cells in order. On a free T4 GPU this extracts all PDFs and
generates the evaluation cases in roughly 20-40 minutes. Results are
downloaded as a zip at the end. Nothing here needs a paid API.

> Note: after the first run, an **on-disk cache** (`data/fact_cache/`) makes
> any repeat run near-instant, so you can re-upload a *new* PDF cheaply —
> mirroring the web UI where re-processing the same document is free.


### 0. Connect to a GPU (runtime > Change runtime type > T4 GPU)

In [ ]:
import os
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)
!nvidia-smi | head -3


### 1. Clone the repo & install dependencies

In [ ]:
import sys, subprocess, pathlib
REPO_URL = "https://github.com/Janavee01/grounded_fact_engine.git"
ROOT = pathlib.Path("/content/fact-knowledge-layer")
if not ROOT.exists():
    !git clone {REPO_URL} /content/fact-knowledge-layer
else:
    print("already cloned")
%cd /content/fact-knowledge-layer
!pip install -q -r requirements.txt 2>&1 | tail -1
!apt-get -qq install -y tesseract-ocr >/dev/null 2>&1 && echo "tesseract-ocr installed"


### 2. Install & start Ollama on this Colab VM (no sudo needed)

In [ ]:
import subprocess, time, pathlib, urllib.request
# Install ollama binary (no sudo on Colab)
p = pathlib.Path("/usr/local/bin/ollama")
if not p.exists():
    url = "https://ollama.com/download/ollama-linux-amd64.tgz"
    print("downloading ollama...")
    !curl -fsSL -o /tmp/ollama.tgz {url}
    !tar -xzf /tmp/ollama.tgz -C /usr/local
    print("installed")
!ollama --version


In [ ]:
import subprocess, os, time
# Start the ollama server in the background
env = dict(os.environ)
env["OLLAMA_HOST"] = "127.0.0.1:11434"
# Keep the large model resident so we don't reload between calls
env["OLLAMA_KEEP_ALIVE"] = "1h"

log = open("/tmp/ollama.log", "w")
proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=log, stderr=log)
time.sleep(4)
print("ollama server pid:", proc.pid)
!curl -s --max-time 5 http://127.0.0.1:11434/api/tags | head -c 200


### 3. Pull the model

`qwen3:8b` gives noticeably better JSON/quote fidelity than `:4b` and still fits the T4's 16 GB.

In [ ]:
!ollama pull qwen3:8b
!ollama list | grep qwen3


### 4. Upload PDFs — mirrors the web app's sidebar upload

The web app (`app_ui.py`) lets you upload any PDF through the sidebar, which
the FastAPI `POST /upload` endpoint saves under `data/uploads/` and then
extracts. The widget below does the same; files land in `data/uploads/`.
Skip this cell if you only want to use the starter PDFs already shipped in
the repo (`data/india-macroeconomy/`, `data/delhivery/`).

In [ ]:
import pathlib
from google.colab import files
# OPTIONAL: upload any PDFs you want to test the layer with, exactly like
# the website sidebar. They are saved to data/uploads/ (same as POST /upload).
uploaded = files.upload()
UPLOAD_DIR = pathlib.Path("data/uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
for name, content in uploaded.items():
    (UPLOAD_DIR / name).write_bytes(content)
    print("saved:", UPLOAD_DIR / name)


### 5. Extract facts from every PDF — mirrors the web app's **Process** button

The pipeline runs for the starter PDFs (`data/india-macroeconomy/`,
`data/delhivery/`) and anything you uploaded. One LLM call per page,
passed through the grounding gate (`ground_quote`, min score 80) so every
fact is backed by verbatim source text.

In [ ]:
import os, sys, time, pathlib
sys.path.insert(0, "/content/fact-knowledge-layer")
os.environ["LLM_PROVIDER"] = "ollama"
os.environ["OLLAMA_MODEL"] = "qwen3:8b"

from src.extraction.pdf_extractor import PDFExtractor
from src.extraction.llm_client import LLMClient

llm = LLMClient()   # provider='ollama', model=qwen3:8b, OLLAMA_HOST=127.0.0.1:11434
extractor = PDFExtractor(llm_client=llm, max_workers=2)

base = pathlib.Path("/content/fact-knowledge-layer/data")
pdfs = sorted(base.rglob("*.pdf"))
print("found", len(pdfs), "pdfs")
for p in pdfs:
    t = time.time()
    facts = extractor.extract_facts(str(p), source_document=p.name)
    print(f"  {p.name}: {len(facts)} facts in {time.time()-t:.0f}s", flush=True)


### 6. Persist facts & view stats — mirrors the web app's **Stats** tab

Facts are written to the same SQLite database (`data/facts.db`) the local app
uses, matching `POST /upload`. The summary below matches the `GET /stats`
response the UI renders.

In [ ]:
import json
from src.storage.database import Database

db = Database(db_path="data/facts.db")
facts_all = []
for p in pdfs:
    facts = extractor.extract_facts(str(p), source_document=p.name)
    facts_all.extend(facts)
    for f in facts:
        db.save_fact(f)

documents = sorted({f.source_document for f in facts_all})
fact_types = {}
for f in facts_all:
    t = f.fact_type.value
    fact_types[t] = fact_types.get(t, 0) + 1

print("total_facts:", len(facts_all))
print("total_documents:", len(documents))
print("documents:", documents)
print("fact_types:", fact_types)


### 7. Run comparisons — mirrors the web app's **Compare** tab

Cross-document pairs are filtered by the candidate gate, then classified by
the LLM as `corroborates` / `contradicts` / `reconciled` (or dropped as
`unrelated`). The web app shows a live progress bar via `GET /compare/progress`;
here the same progress callback prints inline.

In [ ]:
import sys, json, time
from src.comparison.comparator import FactComparator

comparator = FactComparator(llm_client=llm)

def on_progress(completed, total, stage):
    print(f"\r[{stage}] {completed}/{total}", end="", flush=True)

comparisons = comparator.compare_facts(facts_all, progress_callback=on_progress)
print("\ncomparisons:", len(comparisons))

# Group exactly like the FastAPI GET /compare response
corroborations = [c for c in comparisons if c.relationship == "corroborates"]
contradictions = [c for c in comparisons if c.relationship == "contradicts"]
reconciled = [c for c in comparisons if c.relationship == "reconciled"]
insufficient = [c for c in comparisons if c.relationship == "insufficient_context"]

result = {
    "total_comparisons": len(comparisons),
    "corroborations": [c.model_dump() for c in corroborations],
    "contradictions": [c.model_dump() for c in contradictions],
    "reconciled": [c.model_dump() for c in reconciled],
    "insufficient_context": [c.model_dump() for c in insufficient],
    "summary": {
        "corroborations_count": len(corroborations),
        "contradictions_count": len(contradictions),
        "reconciled_count": len(reconciled),
        "insufficient_context_count": len(insufficient),
    },
}
json.dump(result, open("comparisons.json", "w"), indent=2, default=str)
print("summary:", result["summary"])


### 8. The four required cases — mirrors the web app's **Demo Cases** tab

Each case prints the model's reasoning (`explanation`) *and* the verbatim
source evidence (`source_snippet`) for both facts, exactly like the UI's
Demo Cases / Facts tabs show them.

In [ ]:
import json
data = json.load(open("comparisons.json"))
by_id = {}
for f in facts_all:
    by_id[f.id] = f

def show(rel, title):
    hits = data.get(rel, [])
    print(f"=== {title} ({len(hits)}) ===")
    for c in hits[:3]:
        print("  EXPL:", c.get("explanation", "")[:300])
        print("  CONF:", c.get("confidence"))
        fa, fb = by_id.get(c.get("fact1_id")), by_id.get(c.get("fact2_id"))
        if fa is not None:
            print("  SOURCE A:", fa.source_document)
            print("    SNIPPET:", fa.source_snippet[:200])
        if fb is not None:
            print("  SOURCE B:", fb.source_document)
            print("    SNIPPET:", fb.source_snippet[:200])
        if c.get("context_notes"):
            print("  NOTES:", c["context_notes"])
        print()

# Case 1 — Corroboration
show("corroborations", "Case 1 — Corroboration (same fact, different wording)")

# Case 2 — Contradiction
show("contradictions", "Case 2 — Contradiction (same metric, incompatible values)")

# Case 3 — Reconciliation
show("reconciled", "Case 3 — Reconciliation (apparent conflict explained by units/scope)")

# Case 4 — Extraction / reasoning failure handled honestly
print("=== Case 4 — Extraction failure handled honestly ===")
print("The grounding gate (RapidFuzz, min score 80) rejects any fact whose")
print("verbatim_quote does not match the raw PDF text; rejections are logged,")
print("never fatal. OCR pages are tagged context['text_source']='ocr'. Comparisons")
print("with insufficient context are labelled 'insufficient_context', not forced.")
print()
for c in data.get("insufficient_context", [])[:3]:
    fa, fb = by_id.get(c.get("fact1_id")), by_id.get(c.get("fact2_id"))
    print("  EXPL:", c.get("explanation", "")[:300])
    if fa is not None:
        print("  SOURCE A:", fa.source_document, "| SNIPPET:", fa.source_snippet[:150])
    if fb is not None:
        print("  SOURCE B:", fb.source_document, "| SNIPPET:", fb.source_snippet[:150])
    print()


### 9. Package everything

Downloads a zip with facts, comparisons, stats, and the cache so you can
attach it to the assignment or use it in the demo video — same artifacts the
web app serves through `/facts`, `/compare`, and `/stats`.

In [ ]:
import json, os
from google.colab import files
from src.storage.database import Database
db = Database("data/facts.db")
facts = db.get_all_facts()
json.dump(facts, open("facts_export.json", "w"), indent=2, default=str)
!zip -q -r results.zip facts_export.json comparisons.json data
print("zip created (~%d KB)" % (os.path.getsize("results.zip")//1024))
files.download("results.zip")
print("downloaded. Done!")


### Done

- The full web-app flow ran: **upload → extract → stats → compare → demo cases**.
- Results live in `data/fact_cache/` and `data/facts.db`, so the local app can
  reuse them and repeat runs are near-instant.
- Scanned/image-only PDFs are handled automatically: pages with no selectable
  text are OCR'd with Tesseract (installed above) and their facts are tagged
  `context["text_source"] = "ocr"`.
